In [1]:
# ============================================================
# WC Benchmark Dataset
# NLP + K-Means Clustering
# Jupyter Notebook
# ============================================================

# If required, uncomment and run:
# !pip install pandas numpy scikit-learn matplotlib seaborn nltk

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLP resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")


# ============================================================
# 2. LOAD THE DATASET
# ============================================================

file_path = "wc_benchmark.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())


# ============================================================
# 3. INSPECT THE DATASET
# ============================================================

print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:", df.duplicated().sum())


# ============================================================
# 4. FIND THE TEXT COLUMN
# ============================================================

# Change this manually if your text column has a known name.
possible_text_columns = [
    "text",
    "review",
    "comment",
    "content",
    "description",
    "sentence",
    "document"
]

text_column = None

for col in possible_text_columns:
    if col in df.columns:
        text_column = col
        break

if text_column is None:
    # Automatically select the object/string column
    text_columns = df.select_dtypes(include=["object"]).columns

    if len(text_columns) > 0:
        text_column = text_columns[0]
    else:
        raise ValueError(
            "No text column was found. Please set text_column manually."
        )

print("Text column selected:", text_column)


# ============================================================
# 5. REMOVE MISSING TEXT
# ============================================================

df = df.dropna(subset=[text_column]).copy()

df[text_column] = df[text_column].astype(str)

print("Dataset shape after removing missing text:", df.shape)


# ============================================================
# 6. NLP TEXT CLEANING
# ============================================================

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def clean_text(text):
    """
    Clean text using basic NLP preprocessing:
    - lowercase
    - remove URLs
    - remove punctuation
    - remove numbers
    - tokenize
    - remove stopwords
    - lemmatise words
    """

    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove HTML
    text = re.sub(r"<.*?>", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenise
    words = text.split()

    # Remove stopwords and lemmatise
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 2
    ]

    return " ".join(words)


# Apply NLP preprocessing
df["clean_text"] = df[text_column].apply(clean_text)

display(df[[text_column, "clean_text"]].head(10))


# ============================================================
# 7. REMOVE EMPTY DOCUMENTS
# ============================================================

df = df[df["clean_text"].str.strip() != ""].copy()

print("Dataset shape after NLP cleaning:", df.shape)


# ============================================================
# 8. CONVERT TEXT INTO TF-IDF FEATURES
# ============================================================

tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = tfidf.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", X.shape)


# ============================================================
# 9. FIND THE BEST NUMBER OF CLUSTERS
# ============================================================

# Test different values of K
k_values = range(2, 11)

silhouette_scores = []

for k in k_values:

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X)

    score = silhouette_score(X, labels)

    silhouette_scores.append(score)

    print(f"K = {k}, Silhouette Score = {score:.4f}")


# ============================================================
# 10. PLOT SILHOUETTE SCORES
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Selecting the Best Number of K-Means Clusters")

plt.xticks(list(k_values))
plt.grid(True)

plt.show()


# ============================================================
# 11. SELECT BEST K
# ============================================================

best_k = list(k_values)[np.argmax(silhouette_scores)]

print("Best number of clusters:", best_k)
print(
    "Best silhouette score:",
    round(max(silhouette_scores), 4)
)


# ============================================================
# 12. TRAIN FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

print("K-Means clustering completed.")


# ============================================================
# 13. CLUSTER SIZE
# ============================================================

cluster_counts = df["cluster"].value_counts().sort_index()

print("\nNumber of documents in each cluster:")
print(cluster_counts)


# ============================================================
# 14. VISUALISE CLUSTER SIZES
# ============================================================

plt.figure(figsize=(10, 5))

sns.barplot(
    x=cluster_counts.index,
    y=cluster_counts.values
)

plt.xlabel("Cluster")
plt.ylabel("Number of Documents")
plt.title("Documents per K-Means Cluster")

plt.show()


# ============================================================
# 15. FIND IMPORTANT WORDS IN EACH CLUSTER
# ============================================================

terms = tfidf.get_feature_names_out()

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

print("\n" + "=" * 60)
print("TOP WORDS IN EACH CLUSTER")
print("=" * 60)

for cluster_number in range(best_k):

    top_words = [
        terms[index]
        for index in order_centroids[cluster_number, :15]
    ]

    print(f"\nCluster {cluster_number}:")
    print(", ".join(top_words))


# ============================================================
# 16. ADD CLUSTER LABELS
# ============================================================

df["cluster_label"] = df["cluster"].apply(
    lambda x: f"Cluster {x}"
)

display(
    df[
        [text_column, "clean_text", "cluster", "cluster_label"]
    ].head(20)
)


# ============================================================
# 17. PCA VISUALISATION
# ============================================================

# Reduce TF-IDF dimensions to 2D
pca = PCA(n_components=2, random_state=42)

X_pca = pca.fit_transform(X.toarray())

df["PCA1"] = X_pca[:, 0]
df["PCA2"] = X_pca[:, 1]


# ============================================================
# 18. PLOT K-MEANS CLUSTERS
# ============================================================

plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=df,
    x="PCA1",
    y="PCA2",
    hue="cluster",
    palette="tab10",
    s=60,
    alpha=0.7
)

plt.title("K-Means Clustering of WC Benchmark Dataset")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")

plt.legend(title="Cluster")

plt.show()


# ============================================================
# 19. DISPLAY EXAMPLES FROM EACH CLUSTER
# ============================================================

for cluster_number in range(best_k):

    print("\n" + "=" * 80)
    print(f"CLUSTER {cluster_number}")
    print("=" * 80)

    cluster_data = df[df["cluster"] == cluster_number]

    display(
        cluster_data[
            [text_column, "clean_text"]
        ].head(5)
    )


# ============================================================
# 20. SAVE RESULTS
# ============================================================

output_file = "wc_benchmark_kmeans_results.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"\nResults saved to: {output_file}")


# ============================================================
# 21. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print("Original dataset size:", len(df))
print("Number of TF-IDF features:", X.shape[1])
print("Optimal number of clusters:", best_k)
print("Silhouette score:", round(max(silhouette_scores), 4))

print("\nCluster distribution:")
print(df["cluster"].value_counts().sort_index())

print("\nTop terms per cluster:")

for cluster_number in range(best_k):

    top_words = [
        terms[index]
        for index in order_centroids[cluster_number, :10]
    ]

    print(
        f"Cluster {cluster_number}: "
        + ", ".join(top_words)
    )

Dataset loaded successfully!
Shape: (9, 5)


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,WC,TP,FP,FN,NB
0,AC,3541,276,832,325
1,AR,44,3,15,4
2,AS,1922,141,647,188
3,EC,47,4,5,11
4,ES,487,42,86,68


Column names:
['WC', 'TP', 'FP', 'FN', 'NB']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   WC      9 non-null      str  
 1   TP      9 non-null      int64
 2   FP      9 non-null      int64
 3   FN      9 non-null      int64
 4   NB      9 non-null      int64
dtypes: int64(4), str(1)
memory usage: 510.0 bytes

Missing values:
WC    0
TP    0
FP    0
FN    0
NB    0
dtype: int64

Number of duplicate rows: 0
Text column selected: WC
Dataset shape after removing missing text: (9, 5)


/var/folders/88/w4w1n8l12kd_z42_6mrnndmw0000gn/T/ipykernel_3644/3643908281.py:90: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include=["object"]).columns


,WC,clean_text
0,AC,
1,AR,
2,AS,
3,EC,
4,ES,
5,IN,
6,MC,
7,MS,
8,NI,


Dataset shape after NLP cleaning: (0, 6)


ValueError: empty vocabulary; perhaps the documents only contain stop words